# Clasificador de Features IR — MobileNetV2 TFLite INT8

Entrena un clasificador por denominacion usando MobileNetV2.
Exporta a TensorFlow Lite con cuantizacion INT8 para RPi Zero 2W.

Dataset: segmentos_ir_billetes (crops por feature type)

In [ ]:
import os, shutil, random, tempfile
import numpy as np
from pathlib import Path
from collections import Counter

import tensorflow as tf
from tensorflow.keras import layers, models, applications
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

---
## 2. Configuracion
---

In [ ]:
SEGMENTS_DIR = Path("../Data/raw/imagenes_ir/segmentos_ir_billetes")
MODELS_DIR = Path("../Models_rpi")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DENOMS = ['10', '20', '50', '100', '200']
IMG_SIZE = 96  # reducido para RPi Zero
BATCH_SIZE = 32
EPOCHS = 20

# Clases por denominacion (las que hay en el dataset)
CLASS_MAP = {
    '10': ['animal_10bs', 'ir_10bs', 'personaje_10bs', 'serie_a', 'valor_10bs'],
    '20': ['animal_20bs', 'ir_20bs', 'personaje_20bs', 'serie_a', 'valor_20bs'],
    '50': ['animal_50bs', 'ir_50bs', 'personaje_50bs', 'serie_a', 'valor_50bs'],
    '100': ['animal_100bs', 'ir_100bs', 'personaje_100bs', 'serie_a', 'valor_100bs'],
    '200': ['animal_200bs', 'ir_200bs', 'personaje_200bs', 'serie_a', 'valor_200bs'],
}

print(f"Dataset: {SEGMENTS_DIR}")
print(f"Modelos: {MODELS_DIR}")

---
## 3. Preparar dataset por denominacion
---

In [ ]:
def prepare_denom_dataset(denom):
    """Copia crops de una denominacion a estructura flat para ImageDataGenerator."""
    tmp = Path(tempfile.mkdtemp()) / denom
    target_classes = CLASS_MAP[denom]
    
    counts = {}
    for cls in target_classes:
        (tmp / cls).mkdir(parents=True, exist_ok=True)
        
    for side in ['anverso', 'reverso']:
        for cls in target_classes:
            # El dataset tiene: 10_Bs/anverso/personaje_10bs/*.png
            # o: 10_Bs/anverso/serie_a_10bs/*.png
            src_dir = SEGMENTS_DIR / f"{denom}_Bs" / side
            if not src_dir.exists():
                continue
            for sub in src_dir.iterdir():
                if not sub.is_dir():
                    continue
                # Match: personaje_10bs, serie_a, etc.
                sub_cls = None
                for tc in target_classes:
                    if sub.name.startswith(tc) or sub.name == tc:
                        sub_cls = tc
                        break
                if sub_cls is None:
                    continue
                for img_path in sub.glob("*.png"):
                    dst = tmp / sub_cls / img_path.name
                    shutil.copy2(str(img_path), str(dst))
                    counts[sub_cls] = counts.get(sub_cls, 0) + 1
    
    for cls, n in sorted(counts.items()):
        print(f"  {cls}: {n}")
    print(f"  Total: {sum(counts.values())}")
    return tmp, counts

# Ver todas las denominaciones
for denom in DENOMS:
    print(f"\n{denom} Bs:")
    tmp, counts = prepare_denom_dataset(denom)
    shutil.rmtree(tmp.parent)

---
## 4. Entrenar MobileNetV2 por denominacion → TFLite
---

In [ ]:
def build_model(num_classes):
    base = applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
        alpha=0.35
    )
    base.trainable = False
    model = models.Sequential([
        layers.Rescaling(1./255, input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def representative_dataset_gen(generator):
    """Generador para cuantizacion INT8."""
    def gen():
        for _ in range(100):
            x, _ = next(generator)
            yield [x.astype(np.float32)]
    return gen

for denom in DENOMS:
    print(f"\n{'='*50}")
    print(f"ENTRENANDO {denom} Bs")
    print(f"{'='*50}")
    
    tmp, counts = prepare_denom_dataset(denom)
    num_classes = len(counts)
    
    datagen = ImageDataGenerator(
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.15,
        horizontal_flip=False,
        validation_split=0.2
    )
    
    train_gen = datagen.flow_from_directory(
        str(tmp), target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE, class_mode='categorical',
        subset='training', shuffle=True
    )
    
    val_gen = datagen.flow_from_directory(
        str(tmp), target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE, class_mode='categorical',
        subset='validation', shuffle=False
    )
    
    print(f"Train: {train_gen.samples}, Val: {val_gen.samples}")
    print(f"Clases: {list(train_gen.class_indices.keys())}")
    
    model = build_model(num_classes)
    
    model.fit(
        train_gen, validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3),
        ]
    )
    
    # Evaluar
    loss, acc = model.evaluate(val_gen, verbose=0)
    print(f"Val accuracy: {acc:.4f}")
    
    # Exportar TFLite INT8
    model_path = MODELS_DIR / f"classifier_{denom}bs"
    model.save(str(model_path) + ".h5")
    
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset_gen(train_gen)
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    
    tflite_model = converter.convert()
    tflite_path = model_path.with_suffix('.tflite')
    with open(tflite_path, 'wb') as f:
        f.write(tflite_model)
    
    size_kb = len(tflite_model) / 1024
    print(f"TFLite INT8: {size_kb:.1f} KB -> {tflite_path.name}")
    
    shutil.rmtree(tmp.parent)

print("\n Todos los modelos exportados a", MODELS_DIR)